# 00 — Environment check on Kaggle

**Purpose.** Record the real machine (GPU name, Python, PyTorch, CUDA, disk) and prove that MediaPipe 0.10.35 installs and runs on a real frame of one of our videos — before any long run.

**Kaggle settings.** Accelerator **GPU** (T4 or P100, whichever is offered) · Internet **On**.

**Inputs.** The private dataset `bdsl-k400` only.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `outputs/env/env_report.json`, `outputs/env/mediapipe_route.json`.

**Expected run time** [ESTIMATE, not measured yet]. about 3–6 minutes (pip install + model download + one frame).

**What to check.** The last cell says **NOTEBOOK RESULT: PASS**; `GPU visible` is PASS; `MEDIAPIPE: OK via main` (or via fallback). If MediaPipe FAILS on both routes, stop and paste the output to Claude before going on.

> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "00_env_check_kaggle"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

## Step 1 — GPU, Python, CUDA, disk (real values, saved to `outputs/env/env_report.json`)

In [ ]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
except FileNotFoundError:
    print("nvidia-smi not found: no NVIDIA GPU in this session")
rc_env = S.k400("env-report")

## Step 2 — MediaPipe 0.10.35: install and run on one real video frame
Main route: `pip install mediapipe==0.10.35 opencv-contrib-python==5.0.0.93` into this Python. If that fails, the pinned **fallback** is tried automatically: a separate Python 3.12 made with `uv`, with the desktop's exact versions. Notebook 02 uses whichever route works.

In [ ]:
rc_mp = S.k400("mp-setup")

## Step 3 — Verdict

In [ ]:
env = json.load(open(S.state_root / "outputs/env/env_report.json"))
mp = json.load(open(S.state_root / "outputs/env/mediapipe_route.json"))
gpus = env["device"]["gpu_names"]
print("Python        :", env["python"])
print("PyTorch / CUDA:", env["device"]["torch"], "/", env["device"]["torch_cuda_version"])
print("GPU           :", gpus or "NONE - turn on the GPU accelerator (GUIDE step 5)")
print("Free disk     :", env["disk_free_gb_working"], "GB in /kaggle/working")
print("MediaPipe     :", mp["route"], "->", mp.get("python"))
print("Probe         :", mp.get("probe"))
if not mp["ok"]:
    print("\nBOTH ROUTES FAILED. Do not run notebook 02. Paste this whole output to Claude. Backup option: extract features on the desktop (kaggle/README.md, section Local extraction).")

In [ ]:
S.finish(checks=[
    ("GPU visible to PyTorch", bool(gpus)),
    ("MediaPipe 0.10.35 runs on a real frame", mp["ok"]),
    ("model file SHA-256 equals configs/base.yaml", mp["model"]["ok"]),
])